# 18 pamoka: AI agentų apsauga naudojant kriptografinius kvitus

## Praktinis sąsiuvinis

Šiame sąsiuvinyje atliekamos keturios užduotys:

1. **Pasirašykite pirmąjį kvitą** už agento įrankio iškvietimą ir patikrinkite jį.
2. **Pakeiskite kvitą** ir stebėkite, kaip patikra nepavyksta.
3. **Sukurkite trijų kvitų grandinę** ir patvirtinkite grandinės vientisumą.
4. **Apvyniokite Microsoft Agent Framework įrankio iškvietimą**, kad kiekvienas veiksmas išleistų kvitą.

Visos kriptografinės priemonės importuojamos iš gerai prižiūrimų bibliotekų (`pynacl` Ed25519, `jcs` RFC 8785 kanoniniam JSON, `hashlib` iš Python standartinės bibliotekos SHA-256). Pats kvito logikos kodas yra paprastas Python, kurį galite skaityti ir keisti.

Paleiskite ląsteles tvarka. Kiekviena dalis yra trumpa ir savarankiška.


## Diegimas

Įdiekite dvi priklausomybes. Abi turi leidžiamąsias licencijas (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Pagalbinės priemonės

Šie du pagalbiniai įrankiai atlieka base64url kodavimą (be pabaigos) ir SHA-256 maišos funkciją atsitiktiniams objektams. Jie leidžia likusiai užrašų knygelės daliai sutelkti dėmesį ties pačios kvito logika.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## 1 skyrius: Pasirašykite savo pirmąjį kvitą

Įsivaizduokite, kad mūsų **Contoso Travel** agentas ką tik paieškojo skrydžių iš Sidnėjaus į Los Andželą klientui. Norime įrašyti vartotojo pareiškimą apie šį įrankio skambutį, kad būsimasis auditas galėtų jį patikrinti pagal atskirai prisegtą leidėjo raktą.

### 1.1 veiksmas: Sugeneruokite pasirašymo raktą

Gamyboje agento pasirašymo raktas gyventų aparatinės įrangos saugumo modulyje (HSM), Azure Key Vault arba panašioje saugomoje saugykloje. Šiam pamokos tikslui sugeneruojame naują raktą atmintyje.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### 1.2 žingsnis: Sukurkite kvito apkrovą

Apkrova apima viską, ką norime, kad kvitas patvirtintų: kas veikė, kokį įrankį, su kokiais argumentais, kas grįžo, pagal kokią politiką ir kada. Argumentus ir rezultatą maišome (hash), o ne įtraukiame juos tiesiogiai, kad kvitas neišleistų jautrios informacijos.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### 1.3 veiksmas: Pasirašykite ir surinkite kvitą

Trys veiksmai:

1. Kanonizuokite apkrovą naudodami JCS, kad dvi to paties loginio kvito kūrimo įgyvendinimo versijos duotų bitų identiškus baitus.
2. Tiesiogiai pasirašykite kanonizuotus baitus naudodami Ed25519 privatų raktą. PureEdDSA viduje maišo žinutę, todėl papildomas išankstinis maišas pakeistų protokolą.

Parašas tada pridedamas prie originalios apkrovos, kad būtų sukurtas galutinis kvitas.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### 1.4 žingsnis: Patvirtinkite kvitą

Patvirtinimas atlieka procesą atvirkščiai. Mes pašaliname parašą, perskaičiuojame kanoninius baitus ir patikriname parašą pagal leidėjo viešąjį raktą.

Patvirtinimas vyksta neprisijungus: nėra jokių paslaugų kvietimui ir nėra tinklo prieigos. Vis dėlto, be kvito, reikia dar vieno dalyko: leidėjo viešojo rakto, gauto per kanalą, kuriuo auditorius jau pasitiki, pavyzdžiui, per JWK rinkinį, kurį leidėjas paskelbia, arba iš anksto nustatytą reikšmę. Laukas `public_key` kvito viduje negali atlikti tokios funkcijos, nes kas rašo kvitą - tas rašo ir tą lauką. 2 skyrius parodo klastojimą, kurį šis sprendimas užkerta kelią.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Turėtumėte matyti `Receipt is valid: True`, `Pre-hashed receipt valid: False` ir `Malformed receipts rejected: True`. Teigiamas atvejis įrodo, kad tiesioginio JCS pasirašymo kelias veikia; neigiamas kontrolinis atvejis daro parašo srities taisyklę vykdomą, o ne palieka ją kaip tekstą; paskutinė eilutė rodo neteisingos formos įvestį, grąžinančią `False` vietoj išmetimo išimties.


## 2 skyrius: Pakeisti kvitą

Visa kvitų paskirtis yra ta, kad jie būtų aptinkamai pakeisti. Įrodykime tai.

Mes pakeisime tik vieną kvito ženklą ir stebėsime, kaip patvirtinimas nepavyksta.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Kas ką tik įvyko?

Kai pakeitėme `policy_id`, pasikeitė kanoniniai baitai. Parašas (kuris buvo virš originalių kanoninių baitų) nebeatsitinka. Tikrinimas teisingai grąžina `False`.

Nėra būdo pakeisti pasirašytą lauką, tai yra ką nors už `signature` objekto ribų, ir vis tiek, kad kvitas būtų patvirtintas pagal leidėjo viešąjį raktą, nebent užpuoliko rankose yra leidėjo privatus raktas. Pats `signature` objektas nėra pasirašytas: `sig` yra parašas, ir `verify_receipt` ignoruoja `public_key`, taigi vien tos srities redagavimas nieko nekeičia. Tol, kol privatus raktas yra saugykloje, o tikrintojai naudoja leidėjo nepriklausomai paskelbtą raktą, pakeisti pasirašytą lauką ir paslėpti tai neįmanoma. Ši sąlyga yra svarbi: kitas langas rodo pakeistą kvitą, kuris puikiai patikrina kitą raktą.

Išbandykite patys: pakeiskite `tool_name` arba `agent_id` arba `timestamp` aukščiau esančiame lange ir paleiskite iš naujo. Kiekvienas pakeitimas pasirašytame lauke sukuria neteisingą kvitą.


### Sugadinimas, kurio parašas nepagauna

Aukščiau esantis užpuolikas neperrašė parašo. Atsargesnis perkurtų: pakeistų kvitą,
pasirašytų savo sugeneruotu raktu ir įdėtų tą raktą į
`public_key` laukelį. Rezultatas yra kvitas, kurio parašas yra visiškai tikras
jo turiniui.

Paleiskite tai ir pažiūrėkite, kuris iš dviejų patikrinimų tai pagauna.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Turėtumėte matyti `True` tada `False`. Trys `assert` eilutės daro tai
patikrinimu: ląstelė išmeta klaidą, jei klastotė kada nors patvirtinama pagal
išdavimo rakto.

Klastotė yra viduje suderinta. Jos parašas tikrai atitinka jos
turinį ir raktą, kurį ji neša, todėl tikrintuvas, kuris skaito raktą iš
kvito, praneša, kad jis galioja. Nėra nieko negerai su kvitu, nes
niekas jame nėra nesuderinama.

Vienintelis dalykas, kuris ją pagauna, yra tikrinimas pagal raktą, kurio
puolėjas negalėjo paveikti. Štai kodėl `verify_receipt` paima raktą kaip argumentą: 
funkcija su parašu `verify_receipt(receipt)` negali būti saugiai parašyta,
nes vienintelis raktas, kurį ji galėtų pasiekti, yra tas, kurį pateikė puolėjas.

Tas pats kvitas išsaugomas kaip `sample_receipts/04_resigned_receipt.json`, jei norite jį
apžiūrėti už užrašų knygelės ribų.


## 3 skyriaus dalis: Susiekite kvitus tarpusavyje

Vienas kvitas saugo vieną veiksmą. Dauguma agentų atlieka daugybę veiksmų. Kad visa seka būtų saugi nuo klastojimo, mes susiejame kiekvieną kvitą su ankstesniu, į naujo kvito duomenis įtraukdami ankstesnio kvito hash reikšmę.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Jei kas nors pašalina arba pakeičia kvitų tvarką pateiktoje sekoje, kita grandis nutrūksta. Apkarpymo ar pasirašančiojo kontroliuojamo priedo perrašymo aptikimui taip pat reikalinga užfiksuota dabartinė galva, išorinis kontrolinis taškas arba autentiška populiacijos įsipareigojimo priemonė.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Dabar sulaužykite grandinę triukšmaujant vidurinį kvitą ir persitikrinkite. Sugadintas kvitas nepraeina parašo patikrinimo, IR kitas kvitas nepraeina grandinės nuorodos patikrinimo (nes jo `previous_receipt_hash` nebeatitinka pakeisto vidurinio kvito maišos).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Kvitas 0 vis dar patikrina (jis nebuvo pakeistas ir neturi priklausomo ankstesnio kvito). Kvitas 1 nepraeina parašo patikrinimo, nes mes pakeitėme `tool_args_hash`. Kvitas 2 nepraeina grandinės nuorodos patikrinimo, nes jo `previous_receipt_hash` buvo apskaičiuota pagal originalų (dabar pakeistą) kvitą 1.

Net jei užpuolikas persirašytų pakeistą kvitą 1, naujas parašas nepraeina pagal išdavimo rakto patikrą, nebent jis turi išdavimo privataus rakto raktą (persirašymas savo raktu nepavyksta, kaip parodyta 2 skiltyje), ir grandinės nuorodos neatitikimas kvite 2 vis tiek atskleistų klastojimą. Norėdamas paslėpti pakeitimą, užpuolikas turėtų persirašyti kiekvieną kvitą nuo pakeitimo taško, o tam reikalingas išdavimo privataus rakto raktas.


## 4 skyrius: Aplink agentų įrankio iškvietimą uždėti kvito pasirašymą

Tikroje diegimo aplinkoje nenorite, kad kiekvienas agento autorius prisimintų iškviesti `make_receipt`. Jūs norite, kad kvito pasirašymas būtų automatinis kiekvienam įrankio iškvietimui.

Čia yra paprasčiausias šablonas: apvyniojimo klasė, kuri paima bet kokią kviečiamą įrankio funkciją ir grąžina kvitus leidžiančią jos versiją. Tai prisitaiko prie bet kokios agentų sistemos, įskaitant Microsoft Agent Framework (`agent_framework.foundry`).

Jei neturite parengto Microsoft Foundry projekto, žemiau pateikta vietinė imitacija vis tiek demonstruoja šį šabloną.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integracija su Microsoft Agent Framework

Aukščiau pateiktas `ReceiptedTool` apvalkalas yra nepriklausomas nuo sistemos. Norėdami jį naudoti agento, sukurto su Microsoft Agent Framework, viduje, užregistruokite apvalkalą funkciją kaip įrankį. Štai schema (jūs pakeisite mokymo pavyzdį tikruoju Microsoft Foundry įrankio registravimu):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Agentų sistema neturi žinoti nieko apie kvitus. Kvito pasirašymas vyksta aplink įrankį, o ne yra įdėtas į sistemą. Taip jūs pridedate patikimumą esamam agento kodui nepriversdami perrašyti agento.


## Apibendrinimas ir iššūkis plėtimui

Jūs:

- Sugeneravote Ed25519 raktų porą.
- Sukūrėte ir pasirašėte kvitą agento įrankio kvietimui.
- Patikrinote kvitą neprisijungus, naudojant tik leidėjo viešąjį raktą, gautą nepriklausomai nuo kvito.
- Pakeitėte kvito turinį ir pastebėjote patikros nepavykimą.
- Sukūrėte trijų kvitų grandinėlės seką su maišos grandine.
- Pakeitėte grandinės vidurį ir pastebėjote tiek parašo klaidą, tiek grandinės ryšio klaidą.
- Apvyniojote įrankio funkciją su automatinio kvito pasirašymo mechanizmu.

**Iššūkis plėtimui.** Išplėskite kvito schemą `request_id` lauku (UUID paskirstytam sekimui). Atnaujinkite `make_receipt`, kad jį įtrauktų, ir patvirtinkite, kad kvitai vis dar sėkmingai tikrinami nuo pradžios iki pabaigos. Tada modifikuokite lauką po pasirašymo ir įsitikinkite, kad tikrinimas nepavyksta. Tai privers jus suvokti, kaip kiekvienas kanoninėje koduotėje esantis baitas prisideda prie parašo.

**Svarbi riba.** Naudojant fiksuotą raktą ir grandinės kontekstą, kvitai įrodo parašo priskyrimą, turinio integralumą ir ryšį su pateiktu pirmtaku. Jie NEĮRODO vykdymo, šaltinio tikrumo, patikimo laiko, išsamumo, politikos įvertinimo ar teisingumo. Kvito sistema yra pagrindas. Valdymas yra sistema, kurią statote virš jo.

Perskaitykite pamokos README dar kartą turėdami omenyje šią ribą. Dažniausia komandos klaida su kvitais yra manyti, kad „turime kvitus“ reiškia „mes valdome“. Tai nėra tiesa. Kvito sistema daro agento elgesį tikrinamą. Ji jo nepadaro teisingu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
